In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 7: Within-band families. Load fixtures and population payloads (read-only; the API is never called)
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_withinband as B, wo02_analysis as A

fx, pop, prov = P.load_all()
W = P.population_weights(pop)
FX_BY_ID = {q['id']: q for q in fx}
print('provenance:', prov)
print(f'fixtures with a 200: {len(fx)} | population queries with a 200: {len(pop)}')


provenance: {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
fixtures with a 200: 33 | population queries with a 200: 505


In [2]:
# Cell 2 — The eight Within-band statement types (rules in scripts/edop/dashboard/wo02_rules_withinband.py)
for s in B.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print("families (from the catalog's dimension field, checked against the payloads): soil texture, river discharge, wetlands, temperature, elevation, land use")
print("each statement runs on its own items: a family and a side (local or upstream), so rates are shares of those items")


WB-01  Soil texture as one composition (one basin)
      applies to: single basin and one-basin buffer; local and upstream; L6, L8
      template  : Soil texture{ (upstream)}: {n}% clay, {n}% silt, {n}% sand{ (these sum to {n}%, not 100%)}.
WB-02  Soil texture ranks are not independent
      applies to: all scopes with the three rows usable; local and upstream; L6, L8
      template  : Soil texture{ (upstream)}: clay ranks at the {ord} percentile, silt at the {ord} and sand at the {ord}; the three shares are parts of one whole, so these ranks are not independent.
WB-03  River discharge: mean, lowest and highest month
      applies to: single basin, buffer; L6, L8
      template  : River discharge{ in the basin with the highest annual discharge}: annual mean {n} m3/s, from {n} m3/s in the lowest month to {n} m3/s in the highest ({n} times | ; at least one month has no flow).
WB-04  Wetland extent: wetland and open water, and the part that is open water
      applies to: single basin and

In [3]:
# Cell 3 — Fixtures: apply every rule; count where each applies and why not
fdf = A.apply_fixtures(fx, B)
print(A.fixture_table(fdf).to_string())
print()
print('why a statement did not apply (count of items):')
print(A.reason_table(fdf).to_string())


res                 applied  not_applicable
id    scope                                
WB-01 buffer              0              10
      polity              0               8
      single_basin       46               0
WB-02 buffer              6               4
      polity              8               0
      single_basin       46               0
WB-03 buffer              5               0
      polity              0               4
      single_basin       23               0
WB-04 buffer              0              10
      polity              0               8
      single_basin       21              25
WB-05 buffer              0               5
      polity              0               4
      single_basin       11              12
WB-06 buffer              0               5
      polity              0               4
      single_basin       23               0
WB-07 buffer              0               5
      polity              0               4
      single_basin       23     

In [4]:
# Cell 4 — Draft sentences on chosen fixtures (up to 2 per statement type), to read and judge
for qid in ['F01', 'F04', 'F32']:
    g = fdf[(fdf.qid == qid) & fdf.applies]
    print(f"--- {qid}: {FX_BY_ID[qid]['condition']}")
    for sid, gg in g.groupby('id'):
        for _, r in gg.head(2).iterrows():
            print(f"  {sid}  {r['text']}")
    print()


--- F01: arid; exogenous water (Niger); outside_active_domain
  WB-01  Soil texture: 14% clay, 20% silt, 66% sand.
  WB-01  Soil texture (upstream): 23% clay, 21% silt, 55% sand.
  WB-02  Soil texture: clay ranks at the 23rd percentile, silt at the 19th and sand at the 91st; the three shares are parts of one whole, so these ranks are not independent.
  WB-02  Soil texture (upstream): clay ranks at the 67th percentile, silt at the 27th and sand at the 73rd; the three shares are parts of one whole, so these ranks are not independent.
  WB-03  River discharge: annual mean 499 m³/s, from 277 m³/s in the lowest month to 1,050 m³/s in the highest (3.8 times).
  WB-04  Wetland extent: 43% of the basin is wetland or open water (wetland classes including lakes, reservoirs and rivers), 37% is wetland excluding lakes, reservoirs and rivers; the other 6% is lakes, reservoirs and rivers.
  WB-04  Wetland extent (upstream): 13% of the area upstream is wetland or open water (wetland classes including

In [5]:
# Cell 5 — Population sample: how often each statement applies (share of its own items), weighted and unweighted
pdf = A.apply_population(pop, W, B)
print('WEIGHTED by climate-zone size (percent of items):')
print((A.rates(pdf) * 100).round(1).to_string())
print()
print('UNWEIGHTED (percent of items):')
print((A.unweighted_rates(pdf) * 100).round(1).to_string())
print()
print('polities are not in the population sample (5 queries, all fixtures); see Cell 3')


WEIGHTED by climate-zone size (percent of items):
kind   buffer L6  single L6  single L8
id                                    
WB-01        0.3       95.8       91.0
WB-02       44.7       95.8       91.0
WB-03      100.0      100.0      100.0
WB-04        0.3       41.4       31.5
WB-05        0.0       32.9       17.5
WB-06        0.3      100.0      100.0
WB-07        0.3      100.0      100.0
WB-08        0.3      100.0      100.0

UNWEIGHTED (percent of items):
kind   buffer L6  single L6  single L8
id                                    
WB-01        1.0       82.7       77.0
WB-02       43.9       82.7       77.0
WB-03      100.0      100.0      100.0
WB-04        1.0       28.3       19.5
WB-05        0.0       32.7       21.0
WB-06        1.0      100.0      100.0
WB-07        1.0      100.0      100.0
WB-08        1.0      100.0      100.0

polities are not in the population sample (5 queries, all fixtures); see Cell 3


In [6]:
# Cell 6 — The numbers inside the statements (weighted, single basin L6 unless the statement is for areas)
s1 = pdf[(pdf.id == 'WB-01') & pdf.applies].copy()
s1['dev'] = (s1.stat.astype(float) - 100).abs()
print('WB-01 soil texture, how far the three shares are from summing to 100 (percentage points), by side, weighted quantiles:')
for side in ['soil_local', 'soil_upstream']:
    g = s1[(s1.variable == side) & (s1.kind == 'single L6')]
    print(f"  {side}: n {len(g)} | " + ' '.join(f"p{int(q*100)} {P.wquantile(g.dev, g.w, q):.1f}" for q in (0.5, 0.75, 0.9, 0.95)) + f" | share within 1.5: {float((g.w * (g.dev <= 1.5)).sum() / g.w.sum()):.1%}")
print()
print('WB-02 spread of the three soil ranks (highest minus lowest percentile), WB-03 highest-to-lowest month ratio, WB-04 share that is open water, WB-06 temperature range, WB-07 relief, WB-08 cropland + pasture:')
print(A.stat_quantiles(pdf, ['WB-02', 'WB-03', 'WB-04', 'WB-06', 'WB-07', 'WB-08']).round(1).to_string(index=False))
d = pdf[(pdf.id == 'WB-03') & pdf.applies]
print()
print('WB-03: share of discharge rows with at least one month of no flow (ratio undefined), weighted, by kind:',
      {k: round(float((g.w * g.stat.isna()).sum() / g.w.sum() * 100), 1) for k, g in d.groupby('kind')})
w4 = pdf[(pdf.id == 'WB-04') & pdf.applies]
print('WB-04: share of wetland-extent items where group 1 equals group 2 (no lakes, reservoirs or rivers), weighted:',
      {k: round(float((g.w * (g.stat.astype(float) == 0)).sum() / g.w.sum() * 100), 1) for k, g in w4.groupby('kind')})


WB-01 soil texture, how far the three shares are from summing to 100 (percentage points), by side, weighted quantiles:
  soil_local: n 253 | p50 0.0 p75 1.0 p90 1.0 p95 1.0 | share within 1.5: 100.0%
  soil_upstream: n 253 | p50 1.0 p75 1.0 p90 4.0 p95 13.9 | share within 1.5: 75.8%

WB-02 spread of the three soil ranks (highest minus lowest percentile), WB-03 highest-to-lowest month ratio, WB-04 share that is open water, WB-06 temperature range, WB-07 relief, WB-08 cropland + pasture:
   id      kind   n    p10    p25    p50    p75    p90
WB-02 buffer L6  87   25.5   50.1   59.3   68.5   82.1
WB-02 single L6 506   25.6   44.1   61.6   74.0   86.9
WB-02 single L8 154   33.5   48.9   61.1   75.1   89.1
WB-03 buffer L6  78    1.8    3.2    7.6   23.8   58.4
WB-03 single L6 214    2.7    4.4    9.1   25.7   60.9
WB-03 single L8  73    2.9    5.6    9.5   32.9   89.3
WB-04 buffer L6   2    0.0    0.0    0.0    0.0    0.0
WB-04 single L6 173    0.0    0.0    1.0    3.0    7.0
WB-04 single L

In [7]:
# Cell 7 — Sweep: the soil-texture sum note is printed when the three shares are more than 1.5 points from 100 (a display cutoff)
TOLS = [0.5, 1.0, 1.5, 2.0, 5.0, 10.0]
for side in ['soil_local', 'soil_upstream']:
    g = s1[(s1.variable == side) & (s1.kind == 'single L6')]
    print(side, '— weighted share of triples that would carry the note at each tolerance:',
          {f'> {t}': round(float((g.w * (g.dev > t)).sum() / g.w.sum()), 3) for t in TOLS})
print()
print('the other statements state numbers without a cutoff, so there is nothing else to sweep in this family')


soil_local — weighted share of triples that would carry the note at each tolerance: {'> 0.5': 0.309, '> 1.0': 0.0, '> 1.5': 0.0, '> 2.0': 0.0, '> 5.0': 0.0, '> 10.0': 0.0}
soil_upstream — weighted share of triples that would carry the note at each tolerance: {'> 0.5': 0.52, '> 1.0': 0.242, '> 1.5': 0.242, '> 2.0': 0.137, '> 5.0': 0.081, '> 10.0': 0.057}

the other statements state numbers without a cutoff, so there is nothing else to sweep in this family


In [8]:
# Cell 8 — Checks: do the relations hold, what are the upstream soil anomalies, and what is going on with wetland class and extents
c, sums = B.relations(pop + fx)
print('relations (single basin; fixtures and population):')
for k, v in c.items():
    print(f'  {k}: {v}')
print()
cc, sm = B.upstream_soil_anomalies(pop + fx)
print('upstream soil triples more than 1.5 points from 100:', dict(cc))
if sm:
    sm = pd.Series(sm)
    print('   their sums: min', sm.min(), 'median', sm.median(), 'max', sm.max())
print()
zero, pos = B.wetland_zero_by_class(pop + fx)
print('wetland class present but both local extents zero, by class (count):', dict(zero.most_common(8)))
print('wetland class present and group 1 positive, by class (count):', dict(pos.most_common(8)))


relations (single basin; fixtures and population):
  discharge: triples: 429
  discharge: lowest <= mean <= highest: 429
  temperature: triples: 429
  temperature: min <= annual <= max: 429
  elevation: pairs: 429
  elevation: min <= max: 429
  land use: cropland + pasture known: 858
  land use: cropland + pasture <= 100: 858
  wetland class no_data rows: 136
     ... of which both extents zero: 136
  wetland class present rows: 293
     ... of which both extents zero (present class, zero extents): 132
  wetland groups: pairs: 116
  wetland groups: group 1 >= group 2: 116

upstream soil triples more than 1.5 points from 100: {'upstream triples': 353, 'sum more than 1.5 from 100': 111, '   all three zero': 16, '   no zero component (short or over)': 89, '   some component zero': 6}
   their sums: min 0.0 median 96.0 max 98.0

wetland class present but both local extents zero, by class (count): {'Lake': 73, 'Intermittent wetland/lake': 30, 'River': 7, 'Reservoir': 5, 'Freshwater marsh, f

In [9]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Within-band rows)
fxt, rw, ru = A.fixture_table(fdf), A.rates(pdf), A.unweighted_rates(pdf)
rows = []
for s in B.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    row = dict(s, family='Within-band')
    row['fixture_result'] = f"{int(f.applied.sum())} items applied, {int(f.not_applicable.sum())} not applicable ({len(fx)} fixtures)"
    row['population_result'] = ('share of items, weighted: ' + ', '.join(f"{k} {rw.loc[sid, k]:.1%}" for k in rw.columns)
                                + '; unweighted: ' + ', '.join(f"{k} {ru.loc[sid, k]:.1%}" for k in ru.columns) + '; polities not sampled')
    row['sweep_result'] = ('the sum note is printed beyond 1.5 points from 100 (display cutoff; swept in Cell 7)' if sid == 'WB-01' else 'no cutoff')
    rows.append(row)
n_rows = P.upsert_master('Within-band', rows)
print('master table now has', n_rows, 'rows ->', P.MASTER)


master table now has 40 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
